# 6c · Single-call end-to-end reified extraction  (BASELINE)

The **end-to-end baseline** for the staged event-centric pipeline (nb2→nb4→nb6b). Instead of extracting
entities, typing/deduping them, and *then* extracting n-ary nodes + relations over that frozen entity
set, this notebook does it all in **ONE Gemini call from the raw TEXT**: entities (with open free-form
types), n-ary nodes (Event / Observation / Claim), and open `relations`.

It exists to answer *"is the staged decomposition actually better than doing everything in one call?"* —
run it alongside the staged path and compare (context completeness, fragmentation, predicate/type
economy, QA faithfulness, entity precision/recall, truncation rate).

Output is the **same shape** as nb6b's `nary_graph.json`, written to `results/<id>/nary_graph_e2e.json`,
so it flows through the SAME downstream:
- **nb7b** canonicalizes its relations — add `"nary_graph_e2e.json"` to nb7b's input glob/name.
- **nb8** renders it — add `"nary_graph_e2e.json"` to `REL_FILES`.

(Entity *types* here are open too; canonicalizing them is the nb3/nb4-style step — the same gap the
staged n-ary path has. This notebook is the extraction baseline; canonicalization is shared downstream.)

In [ ]:
# --- resolve Phase2_LLM_Extraction root (works whether this notebook lives at the
# project root or under notebooks/<track>/) so results/ & dataset/ paths are stable.
from pathlib import Path
def _find_root(start: Path = None):
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'results').is_dir() and (p / 'dataset').is_dir():
            return p
    return start
ROOT = _find_root()
# make Path writes create missing parent dirs (results/ now has sub-folders like
# two_staged/1_entities/); guarded so re-running the cell never double-wraps.
import pathlib as _pl
if not getattr(_pl.Path.write_text, '_mkparents', False):
    _wt, _wb = _pl.Path.write_text, _pl.Path.write_bytes
    def write_text(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wt(self, *a, **k)
    def write_bytes(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wb(self, *a, **k)
    write_text._mkparents = write_bytes._mkparents = True
    _pl.Path.write_text, _pl.Path.write_bytes = write_text, write_bytes

In [8]:
# --- Config -------------------------------------------------------------------------------
from pathlib import Path

RESULTS_DIR  = ROOT / "results"            # writes results/<id>/nary_graph_e2e.json
DATASET_DIR  = ROOT / "dataset/articles"   # reads <id>.txt (raw text — NO pre-extracted entities)
GEMINI_MODEL = "gemini-2.5-flash"

ARTICLE_ID   = "art1"                      # single article used by the demo cell below
SELECTED_ARTICLES = ["art1", "art2", "art4", "art5"]   # batch set

OUTPUT_NAME  = "e2e/nary_graph.json"       # written to results/<id>/  (distinct from the staged nary_graph.json)

# Articles this run processes: those with a raw <id>.txt.
_available = sorted(p.stem for p in DATASET_DIR.glob("*.txt"))
if SELECTED_ARTICLES is None:
    ARTICLE_IDS = _available
else:
    missing = [a for a in SELECTED_ARTICLES if a not in _available]
    assert not missing, f"no article text for: {missing}"
    ARTICLE_IDS = list(SELECTED_ARTICLES)

print(f"{len(_available)} article texts available; this run will process {ARTICLE_IDS}")

22 article texts available; this run will process ['art1', 'art2', 'art4', 'art5']


In [9]:
# --- Gemini client -----------------------------------------------------------------------
%pip install -q google-genai

import os, json, time, re
from google import genai
from google.genai import types as genai_types
from IPython.display import Markdown, display

GEMINI_API_KEY = "" or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Paste your key in GEMINI_API_KEY above, or set it in the env and restart the kernel."

client = genai.Client(api_key=GEMINI_API_KEY)
last_latency_s = None

MAX_OUTPUT_TOKENS = None   # None -> no cap (model max). This call is BIG (entities+nodes+relations at once).
THINKING_BUDGET   = 0      # thinking OFF so the whole budget goes to the JSON (avoids truncation)

def _loads_lenient(text, resp=None):
    t = (text or "").strip()
    if t.startswith("```"):
        t = re.sub(r"^```[a-zA-Z]*\n?", "", t); t = re.sub(r"\n?```$", "", t).strip()
    try:
        return json.loads(t)
    except json.JSONDecodeError:
        pass
    t2 = re.sub(r",(\s*[}\]])", r"\1", t)
    try:
        return json.loads(t2)
    except json.JSONDecodeError as e:
        fr = None
        try: fr = resp.candidates[0].finish_reason
        except Exception: pass
        raise ValueError(f"Gemini JSON unparseable (finish_reason={fr}; {len(t)} chars). If MAX_TOKENS, "
                         f"the single call TRUNCATED — this is expected to happen sooner than the staged path "
                         f"(bigger output). json error: {e}\n...tail:\n{t[-300:]}") from None

def gemini_json(system_prompt, user_message, model=None, temperature=0):
    global last_latency_s
    cfg = dict(system_instruction=system_prompt, temperature=temperature, response_mime_type="application/json")
    if MAX_OUTPUT_TOKENS is not None:
        cfg["max_output_tokens"] = MAX_OUTPUT_TOKENS
    if THINKING_BUDGET is not None and hasattr(genai_types, "ThinkingConfig"):
        cfg["thinking_config"] = genai_types.ThinkingConfig(thinking_budget=THINKING_BUDGET)
    t0 = time.perf_counter()
    resp = client.models.generate_content(model=model or GEMINI_MODEL, contents=user_message,
                                          config=genai_types.GenerateContentConfig(**cfg))
    last_latency_s = time.perf_counter() - t0
    return _loads_lenient(resp.text, resp)

Note: you may need to restart the kernel to use updated packages.


## 1. Single-call extraction — entities + n-ary nodes + relations, in one pass

One Gemini call takes the raw TEXT and returns everything: entities (with open free-form types and
coreference merged), n-ary nodes (Event / Observation / Claim), and open `relations`. The rules are the
same event-centric rules as nb6b, plus entity extraction and coreference in the same pass — which is
exactly the burden this baseline puts on a single call.

In [ ]:
E2E_SYSTEM_PROMPT = """You are an expert knowledge-graph extraction system. Given a news TEXT and its REFERENCE_DATE (ISO-8601), build a reified, event-centric knowledge graph in ONE pass, with THREE parts.

(1) ENTITIES — the real-world things the TEXT mentions (people, organizations, places, quantities, dates, durations, etc.). Each entity: {"id" ("E1","E2",...), "text" (canonical surface form), "type" (a concise free-form PascalCase class label, as specific as the TEXT warrants), "aliases" (other surface forms for the SAME entity)}. MERGE coreferent mentions into ONE entity; never duplicate an entity.

(2) N-ARY NODES — new reified nodes for the events, observations, and claims the TEXT is about. Kinds:
- "Event": something that happens or is scheduled/forecast to happen.
- "Observation": a stated measurement, statistic, quantity, or quantified change.
- "Claim": a statement explicitly attributed to a source.
Each n-ary node: {"id" ("EV1"/"OBS1"/"CL1",...), "node_kind", "type" (concise free-form PascalCase), "label" (a SHORT handle, a few words — NOT a full sentence), "is_main" (bool), "evidence" (verbatim span(s)), "confidence" (0-1)}.

(3) RELATIONS — open, free-form directed relations {"subject_id","relation_phrase","object_id","evidence","confidence"} where subject_id and object_id are ids of ENTITIES or N-ARY NODES. Invent the concise relation_phrase that best fits (canonicalized downstream); define each unique phrase in relation_definitions.

RULES:
1. Identify the article's MAIN event(s) and set "is_main": true on them.
2. Tie every participant of a stated fact to its n-ary node with a relation, and capture the stable entity-to-entity relations. Aim for ONE connected graph — do not leave a relevant node dangling.
3. Keep every fact's context connected to it: attach all the details that qualify a fact — value, time, place, cause, qualifier, source — to THAT fact's n-ary node, so it forms one connected cluster reachable in a few hops.
4. Keep each n-ary node's label a SHORT handle; do NOT pack facts into it — express the values, places, times, causes and participants as RELATIONS to the ENTITIES instead.
5. Give every Observation and Claim at least one relation to the Event it describes, and relate each Event to other Events it is linked to in the TEXT if applicable (e.g. "part of" / "caused by" / "led to" / "followed").
6. When a breakdown gives several values that each belong to a distinct item (category, group, period), relate EACH value directly to the item it belongs to (value -> item), so every value stays joined to its own item.
7. subject_id and object_id MUST be ids you defined in ENTITIES or N-ARY NODES; never relate a node to itself.
8. Extract every relevant entity; omit only boilerplate (navigation, "related"/"read more" links).
9. Extract only what the TEXT explicitly states. Use REFERENCE_DATE to resolve relative dates where possible.

OUTPUT strict JSON:
{
  "entities": [ {"id","text","type","aliases":[...]} ],
  "nary_nodes": [ {"id","node_kind","type","label","is_main","evidence":[...],"confidence"} ],
  "relations": [ {"subject_id","relation_phrase","object_id","evidence","confidence"} ],
  "relation_definitions": [ {"relation_phrase","definition"} ]
}
"""

def _demojibake(s):
    if isinstance(s, str) and any(m in s for m in ("Ã", "Â", "â€")):
        try: return s.encode("latin-1").decode("utf-8")
        except (UnicodeDecodeError, UnicodeEncodeError): return s
    return s

def _reference_date(text):
    m = re.search(r"^Date:\s*(\d{2})/(\d{2})/(\d{4})", text, re.MULTILINE)
    if not m: return None
    d, mth, y = m.groups(); return f"{y}-{mth}-{d}"

def extract_e2e(aid, write=True):
    """ONE Gemini call: raw TEXT -> entities + n-ary nodes + relations -> results/<id>/nary_graph_e2e.json."""
    text = _demojibake((DATASET_DIR / f"{aid}.txt").read_text())
    ref_date = _reference_date(text)
    out = gemini_json(E2E_SYSTEM_PROMPT, f"REFERENCE_DATE: {ref_date}\n\nTEXT:\n{text}")

    entities   = out.get("entities", [])
    nary_nodes = out.get("nary_nodes", [])
    relations  = out.get("relations", [])
    for e in entities:                                  # normalise: ensure aliases list exists
        e.setdefault("aliases", [])
    node_text = {e["id"]: (e.get("text"), e.get("type")) for e in entities}
    node_text.update({n["id"]: (n.get("label"), n.get("type")) for n in nary_nodes})
    for r in relations:                                 # resolve endpoints (so nb7b/nb8 read subject/object surface+type)
        r["subject_text"], r["subject_type"] = node_text.get(r.get("subject_id"), (None, None))
        r["object_text"], r["object_type"]   = node_text.get(r.get("object_id"), (None, None))

    graph = {"article_id": aid, "reference_date": ref_date, "method": "single_call_e2e",
             "entities": entities, "nary_nodes": nary_nodes,
             "relations": relations, "relation_definitions": out.get("relation_definitions", [])}
    if write:
        p = RESULTS_DIR / aid; p.mkdir(parents=True, exist_ok=True)
        (p / OUTPUT_NAME).write_text(json.dumps(graph, indent=2, ensure_ascii=False))
        print(f"  wrote {p / OUTPUT_NAME}  ({len(entities)} entities, {len(nary_nodes)} n-ary nodes, "
              f"{len(relations)} relations)  [{last_latency_s:.1f}s]")
    return graph

## 2. Inspect the demo article

In [11]:
# g = extract_e2e(ARTICLE_ID)
# by_id = {e["id"]: e.get("text") for e in g["entities"]}
# by_id.update({n["id"]: n["label"] for n in g["nary_nodes"]})
# print("-" * 80)
# print(f'{len(g["entities"])} entities  |  {len(g["nary_nodes"])} n-ary nodes  |  {len(g["relations"])} relations')
# print("-" * 80)
# print("n-ary nodes:")
# for n in g["nary_nodes"]:
#     star = " *MAIN*" if n.get("is_main") else ""
#     print(f'  {n["id"]:<5} {n["node_kind"]:<12} <{n.get("type")}>{star}  "{n["label"]}"')
# print("-" * 80)
# print("relations (first 30):")
# for r in g["relations"][:30]:
#     s = by_id.get(r["subject_id"], r["subject_id"]); o = by_id.get(r["object_id"], r["object_id"])
#     print(f'  {s!r}  --{r["relation_phrase"]}-->  {o!r}')

## 3. Batch — process the selected articles (`ARTICLE_IDS`)

In [12]:
# Single-call extraction for each article. Re-runs overwrite. Compare against the staged nary_graph.json.
for aid in ARTICLE_IDS:
    print("=" * 60, aid)
    extract_e2e(aid)
print(f"\ndone -- {len(ARTICLE_IDS)} article(s) -> results/<id>/{OUTPUT_NAME}")

============================================================ art1
  wrote results/art1/nary_graph_e2e.json  (82 entities, 37 n-ary nodes, 123 relations)  [63.1s]
============================================================ art2
  wrote results/art2/nary_graph_e2e.json  (57 entities, 44 n-ary nodes, 111 relations)  [59.5s]
============================================================ art4
  wrote results/art4/nary_graph_e2e.json  (46 entities, 20 n-ary nodes, 63 relations)  [30.2s]
============================================================ art5
  wrote results/art5/nary_graph_e2e.json  (50 entities, 33 n-ary nodes, 88 relations)  [46.2s]

done -- 4 article(s) -> results/<id>/nary_graph_e2e.json
